<a href="https://colab.research.google.com/github/Roshrol/Rpg-Class/blob/main/report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **RPG Classification**

**Team Members:** Edy Cruz and Layla Robledo  
**GitHub Repository:** https://github.com/Roshrol/Rpg-Class

## Abstract

Our project builds a machine learning model that predicts an RPG character's class based on their stat and skill distribution. We use character attributes such as Strength, Intelligence, Speed, and skills such as Blade, Alchemy, and Sneak as predictors, and the character's RPG class as the target label. We trained a multiclass logistic regression model using cross-entropy loss and evaluated it with training accuracy, test accuracy, and loss. Since there are 21 possible classes, random guessing would only achieve about 5% accuracy, so our goal was to reach around 40–50% accuracy. Our final model reached around 40% test accuracy, meeting our baseline goal.

## Introduction

Role-playing games often ask players to choose a class before they fully understand how different stats and skills affect gameplay. This can be intimidating for new players. Our project addresses this problem by predicting a character's class from their stat distribution. A model like this could eventually help beginners choose a class based on the type of character they want to play.

Previous work has explored similar problems in gaming and machine learning. Joe Macinnes used machine learning and natural language processing to predict Dungeons & Dragons character information from backstories. Jolanta Sliwa explored predictive machine learning in RPG design to study character strength, performance, difficulty, and balance. Louis Khrisna Putera Suryapranata studied personality trait prediction based on game character design using machine learning. These projects relate to ours because they all use character information to predict or evaluate game-related categories.

## Values Statement

The main users of this project would be beginner RPG players who may feel overwhelmed by character creation. A tool based on this model could help players understand what class best matches the stats and skills they are interested in using.

This technology could benefit new players by making RPGs more accessible and less intimidating. It could also benefit game designers who want to create recommendation systems or beginner-friendly character generators.

One possible harm is that the model could oversimplify character creation. Some players enjoy unusual or hybrid builds, and the model might misclassify those characters because they do not follow typical class patterns. This could make players feel like there is only one “correct” way to build a character.

My personal reason for working on this project is that character creation can be intimidating for beginners. A tool like this could make RPGs feel more welcoming and easier to understand.

## Data

Our dataset comes from elderstats.com, where uploaded RPG character information is publicly available. The data was scraped using Python and Playwright. After cleaning, the dataset contains around 280 characters.

Each row represents one character. The target variable is the character's class. The features are 8 main stats, such as Strength, Intelligence, Willpower, Agility, Speed, Endurance, Personality, and Luck, plus 21 skills, such as Blade, Alchemy, Sneak, Restoration, and Marksman.

We removed custom classes because our goal was to focus on premade beginner-friendly classes. We also removed invalid rows with impossible values, such as 0 or 255 in certain stat columns.

In [22]:
import sys
sys.path.append("/content/Rpg-Class/src")
from src.Final_Project import *

ModuleNotFoundError: No module named 'src'

In [23]:
import pandas as pd
import numpy as np
import torch
import pickle
import matplotlib.pyplot as plt

from src.Final_Project import (
    DataPrepPipeline,
    RPG_Model,
    GradientDescentOptimizer,
    cross_entropy_loss,
    accuracy
)

ModuleNotFoundError: No module named 'src'

In [8]:
url = "https://raw.githubusercontent.com/Roshrol/Rpg-Class/refs/heads/main/data/oblivion_characters.csv" #our playwrite info should go here I believe
train = pd.read_csv(url)

print("Number of rows:", train.shape[0])
print("Number of columns:", train.shape[1])

train.head()

Number of rows: 283
Number of columns: 31


,url,class,Armorer,Athletics,Blade,Block,Blunt,Hand to Hand,Heavy Armor,Alchemy,...,Sneak,Speechcraft,Strength,Intelligence,Willpower,Agility,Speed,Endurance,Personality,Luck
0,https://www.elderstats.com/character/8,Thief,100.0,84.0,102.0,50.0,25.0,53.0,100.0,20.0,...,102.0,80.0,100,75,88,100,100,90,90,61
1,https://www.elderstats.com/character/22,Warrior,77.0,43.0,59.0,48.0,32.0,35.0,100.0,100.0,...,14.0,19.0,100,37,41,51,44,100,34,51
2,https://www.elderstats.com/character/31,Rogue,34.0,0.0,30.0,29.0,0.0,0.0,0.0,0.0,...,54.0,0.0,56,40,40,79,73,53,52,60
3,https://www.elderstats.com/character/38,Battlemage,22.0,40.0,100.0,19.0,16.0,5.0,9.0,41.0,...,89.0,11.0,102,102,100,70,90,58,82,51
4,https://www.elderstats.com/character/39,Warrior,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,100,42,100,63,72,58,72,49


## Approach

We used the character's stats and skills as predictors and the character's class as the target. Since the target has more than two possible classes, this is a multiclass classification problem.

Our model is multiclass logistic regression. The model takes the normalized feature values and outputs one score for each possible class. We trained the model using cross-entropy loss because it is appropriate for multiclass classification.

We split the data into an 80% training set and a 20% test set. The data preparation pipeline learned the mean and standard deviation only from the training set, then used those same values to normalize both the training and test sets.

In [ ]:
train_ix = train.sample(frac=0.8, random_state=42).index
test_ix = train.drop(train_ix).index

X_df = train.drop(columns=["class", "url"])
y_df = train["class"]

pipeline = DataPrepPipeline()
pipeline.fit(X_df.loc[train_ix], y_df.loc[train_ix])

X_train = pipeline.transform(X_df.loc[train_ix])
y_train = pipeline.transform_labels(y_df.loc[train_ix])

X_test = pipeline.transform(X_df.loc[test_ix])
y_test = pipeline.transform_labels(y_df.loc[test_ix])

print("Training examples:", X_train.shape[0])
print("Testing examples:", X_test.shape[0])
print("Number of features:", X_train.shape[1])

In [ ]:
num_classes = len(set(y_df))

torch.manual_seed(1)
np.random.seed(1)

model = RPG_Model(X_train.shape[1], num_classes)
opt = GradientDescentOptimizer(model, lr=0.01)

losses = []

for epoch in range(70000):
    q = model.forward(X_train)
    loss = cross_entropy_loss(q, y_train)

    loss.backward()
    opt.step()

    losses.append(loss.item())

    if epoch % 5000 == 0:
        train_acc = accuracy(model, X_train, y_train).item()
        test_acc = accuracy(model, X_test, y_test).item()
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}, Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}")

## Results

The model successfully learned patterns between character stats, skills, and RPG classes. We evaluated the model using cross-entropy loss, training accuracy, and test accuracy. Since there are 21 possible classes, random guessing would only be expected to achieve about 5% accuracy. Our test accuracy was around 40%, which met our original baseline goal of 40–50%.

In [ ]:
final_train_loss = cross_entropy_loss(model.forward(X_train), y_train).item()
train_acc = accuracy(model, X_train, y_train).item()
test_acc = accuracy(model, X_test, y_test).item()

print("Final train loss:", final_train_loss)
print("Train accuracy:", train_acc)
print("Test accuracy:", test_acc)

## Concluding Discussion

Our project worked because the model was able to predict RPG classes much better than random guessing. With 21 possible classes, random guessing would be around 5%, while our model reached around 40% test accuracy. This means the model learned meaningful patterns between stats, skills, and classes.

We met our original goal of reaching around 40–50% accuracy. However, the training accuracy was higher than the test accuracy, which suggests some mild overfitting. This makes sense because our dataset is fairly small and some RPG classes are hybrids that share similar stats and skills.

If we had more time, we would try adding more features such as race and level. We could also experiment with more models and feature engineering to better handle hybrid classes.

## Group Contributions

Edy primarily worked on collecting and scraping the dataset. He used Python and Playwright to gather RPG character information and helped clean invalid characters from the dataset. He also helped manage and organize the GitHub repository.

Layla worked on the data preparation pipeline, including selecting features, normalizing the dataset, transforming labels, and setting up the training process. She also worked on the model training code, accuracy reporting, and organizing the final report notebook.

Both group members experimented with learning rate, number of epochs, and model performance. Both also contributed to the final writing and discussion of the project results.